# LM2011 Sample Post-Refinitiv Runner

Colab wrapper for `lm2011_sample_post_refinitiv_runner.py`. The default paths map the local runner's sample/upstream concepts onto the Drive full-data layout under `/content/drive/MyDrive/Data_LM`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
GIT_REF_ENV_VAR = "NLP_THESIS_GIT_REF"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
DEFAULT_GIT_REF = "main"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but is not the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab."
    )

repo_root = repo_root.resolve()
if IN_COLAB and (repo_root / ".git").exists() and os.environ.get("NLP_THESIS_SKIP_GIT_UPDATE", "0") != "1":
    git_ref = os.environ.get(GIT_REF_ENV_VAR, DEFAULT_GIT_REF)
    fetch_code = subprocess.call(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", git_ref])
    checkout_target = "FETCH_HEAD" if fetch_code == 0 else git_ref
    subprocess.call(["git", "-C", str(repo_root), "checkout", checkout_target])

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("NLP_THESIS_SKIP_PIP_INSTALL", "0") != "1":
    install_target = f"{repo_root}[benchmark]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target])

print({"IN_COLAB": IN_COLAB, "repo_root": str(repo_root), "src_exists": src.exists()})

In [ ]:
from pathlib import Path


def _resolve_colab_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate
    return Path("/content/drive")


def _first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def _check_paths(paths: dict[str, Path], *, require_all: bool = True) -> None:
    rows = []
    missing = []
    for label, path in paths.items():
        exists = path.exists()
        rows.append({"label": label, "path": str(path), "exists": exists})
        if require_all and not exists:
            missing.append(f"{label}: {path}")
    try:
        import polars as pl

        display(pl.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)
    if missing:
        raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

DRIVE_ROOT = _resolve_colab_drive_root()
WORK_ROOT = DRIVE_ROOT / "Data_LM"
SEC_CCM_RUN_ROOT = WORK_ROOT / "results" / "sec_ccm_unified_runner"

# Full-data equivalents of the local runner inputs.
SAMPLE_ROOT = WORK_ROOT
UPSTREAM_RUN_ROOT = SEC_CCM_RUN_ROOT
YEAR_MERGED_DIR = WORK_ROOT / "parquet_data" / "_year_merged"
DAILY_PANEL_PATH = WORK_ROOT / "CRSP_Compustat_data" / "derived_data" / "final_flagged_data_compdesc_added.parquet"
CCM_BASE_DIR = WORK_ROOT / "CRSP_Compustat_data" / "parquet_data"
MATCHED_CLEAN_PATH = SEC_CCM_RUN_ROOT / "sec_ccm_premerge" / "sec_ccm_matched_clean.parquet"
SAMPLE_BACKBONE_PATH = SEC_CCM_RUN_ROOT / "sec_ccm_premerge" / "lm2011_sample_backbone.parquet"
if not SAMPLE_BACKBONE_PATH.exists():
    SAMPLE_BACKBONE_PATH = None
ITEMS_ANALYSIS_DIR = SEC_CCM_RUN_ROOT / "items_analysis"
OUTPUT_DIR = WORK_ROOT / "results" / "lm2011_sample_post_refinitiv_runner"

# Put these files under Data_LM/LM2011_additional_data, or edit this value to a mounted Drive
# folder containing Fin-Neg.txt, LM2011_MasterDictionary.txt,
# F-F_Research_Data_Factors_daily.csv, F-F_Research_Data_Factors.csv,
# F-F_Momentum_Factor.csv, and FF_Siccodes_48_Industries.txt.
ADDITIONAL_DATA_DIR = _first_existing_path(
    WORK_ROOT / "LM2011_additional_data",
    repo_root / "full_data_run" / "LM2011_additional_data",
)

FULL_10K_CLEANING_CONTRACT = "lm2011_paper"  # current | lm2011_paper
MONTHLY_STOCK_PATH = None  # None lets the runner resolve from CCM_BASE_DIR if available.
FF_MONTHLY_WITH_MOM_PATH = None  # Optional override; None auto-builds from monthly FF and momentum CSVs.
TEXT_FEATURE_BATCH_SIZE = 1000

required_paths = {
    "WORK_ROOT": WORK_ROOT,
    "UPSTREAM_RUN_ROOT": UPSTREAM_RUN_ROOT,
    "YEAR_MERGED_DIR": YEAR_MERGED_DIR,
    "DAILY_PANEL_PATH": DAILY_PANEL_PATH,
    "CCM_BASE_DIR": CCM_BASE_DIR,
    "MATCHED_CLEAN_PATH": MATCHED_CLEAN_PATH,
    "ITEMS_ANALYSIS_DIR": ITEMS_ANALYSIS_DIR,
    "ADDITIONAL_DATA_DIR": ADDITIONAL_DATA_DIR,
    "FF_DAILY_CSV": ADDITIONAL_DATA_DIR / "F-F_Research_Data_Factors_daily.csv",
    "FF_MONTHLY_CSV": ADDITIONAL_DATA_DIR / "F-F_Research_Data_Factors.csv",
    "MOMENTUM_MONTHLY_CSV": ADDITIONAL_DATA_DIR / "F-F_Momentum_Factor.csv",
    "FF48_SICCODES": ADDITIONAL_DATA_DIR / "FF_Siccodes_48_Industries.txt",
    "LM_MASTER_DICTIONARY": ADDITIONAL_DATA_DIR / "LM2011_MasterDictionary.txt",
}
_check_paths(required_paths)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "FULL_10K_CLEANING_CONTRACT": FULL_10K_CLEANING_CONTRACT,
    "SAMPLE_BACKBONE_PATH": str(SAMPLE_BACKBONE_PATH) if SAMPLE_BACKBONE_PATH is not None else None,
    "BACKBONE_MODE": "reuse_prebuilt" if SAMPLE_BACKBONE_PATH is not None else "metadata_only_rebuild",
    "TEXT_FEATURE_BATCH_SIZE": TEXT_FEATURE_BATCH_SIZE,
})

In [ ]:
from thesis_pkg.notebooks_and_scripts import lm2011_sample_post_refinitiv_runner as runner

RUN_ARGS = [
    "--sample-root", str(SAMPLE_ROOT),
    "--upstream-run-root", str(UPSTREAM_RUN_ROOT),
    "--additional-data-dir", str(ADDITIONAL_DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--year-merged-dir", str(YEAR_MERGED_DIR),
    "--matched-clean-path", str(MATCHED_CLEAN_PATH),
    "--daily-panel-path", str(DAILY_PANEL_PATH),
    "--items-analysis-dir", str(ITEMS_ANALYSIS_DIR),
    "--ccm-base-dir", str(CCM_BASE_DIR),
    "--full-10k-cleaning-contract", FULL_10K_CLEANING_CONTRACT,
    "--text-feature-batch-size", str(TEXT_FEATURE_BATCH_SIZE),
]
if SAMPLE_BACKBONE_PATH is not None:
    RUN_ARGS.extend(["--sample-backbone-path", str(SAMPLE_BACKBONE_PATH)])
if MONTHLY_STOCK_PATH is not None:
    RUN_ARGS.extend(["--monthly-stock-path", str(MONTHLY_STOCK_PATH)])
if FF_MONTHLY_WITH_MOM_PATH is not None:
    RUN_ARGS.extend(["--ff-monthly-with-mom-path", str(FF_MONTHLY_WITH_MOM_PATH)])

RUN_ARGS

In [ ]:
exit_code = runner.main(RUN_ARGS)
assert exit_code == 0

In [ ]:
import json
import polars as pl

manifest_path = OUTPUT_DIR / "lm2011_sample_run_manifest.json"
print({"manifest_path": str(manifest_path)})
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({"run_status": manifest.get("run_status"), "row_counts": manifest.get("row_counts")}, indent=2))

stages = manifest.get("stages", {})
for stage_name, stage in stages.items():
    print(stage_name, stage.get("status"), stage.get("reason") or "")

for key in ("sample_backbone", "event_panel", "table_iv_results", "table_v_results"):
    path_raw = manifest.get("artifacts", {}).get(key)
    if path_raw:
        path = Path(path_raw)
        print(key, pl.scan_parquet(path).select(pl.len().alias("rows")).collect().item())